# Error Handling in Python

> **Interview Note:** Error handling is one of the most tested topics in Python interviews. Understand not just syntax but *when* and *why* to use each pattern.

---

## 1. Basic `try/except/else/finally`

In [ ]:
def divide(a, b):
    try:
        result = a / b
    except ZeroDivisionError:
        print("Cannot divide by zero")
        return None
    except TypeError as e:
        print(f"Invalid type: {e}")
        return None
    else:
        # Runs ONLY if no exception occurred
        print(f"Result: {result}")
        return result
    finally:
        # ALWAYS runs (cleanup, logging, closing resources)
        print("Division attempt completed")

divide(10, 2)
print("---")
divide(10, 0)
print("---")
divide("10", 2)

### Key Points

- **`except`**: Catches specific exceptions (avoid bare `except:`)
- **`as e`**: Gives access to the exception object
- **`else`**: Runs only if NO exception — useful for code that shouldn't be in `try`
- **`finally`**: Always runs — ideal for cleanup (files, connections, locks)

---

## 2. Exception Hierarchy (Important for Interviews)

In [ ]:
# All exceptions inherit from BaseException
print("BaseException subclasses:")
for cls in BaseException.__subclasses__():
    print(f"  {cls.__name__}")

### Common Exception Types

| Category | Exceptions |
|----------|------------|
| **Base** | `BaseException` (don't catch this), `Exception` (catch this) |
| **Runtime** | `ValueError`, `TypeError`, `KeyError`, `IndexError`, `AttributeError` |
| **I/O** | `FileNotFoundError`, `PermissionError`, `IOError` |
| **Import** | `ImportError`, `ModuleNotFoundError` |
| **Control** | `KeyboardInterrupt`, `SystemExit`, `StopIteration` |


> **Interview Tip:** Never catch `BaseException` — it catches `KeyboardInterrupt` and `SystemExit`, preventing clean shutdown. Catch `Exception` or specific exceptions.

---

## 3. Raising Exceptions

In [ ]:
def validate_age(age):
    if not isinstance(age, int):
        raise TypeError(f"Age must be int, got {type(age).__name__}")
    if age < 0:
        raise ValueError("Age cannot be negative")
    if age > 150:
        raise ValueError("Age seems unrealistic")
    return True

# Test
validate_age(25)
print("Valid")

try:
    validate_age(-5)
except ValueError as e:
    print(f"Caught: {e}")

### Re-raising Exceptions

```python
try:
    risky_operation()
except ValueError:
    log.error("Validation failed")
    raise  # Re-raises the SAME exception with original traceback
```

### Exception Chaining (Python 3+)

```python
try:
    parse_config()
except ParseError as e:
    raise ConfigError("Invalid config") from e  # Preserves original cause
```

`__cause__` and `__context__` attributes track the chain.

---

## 4. Custom Exceptions

In [ ]:
class ModelError(Exception):
    """Base exception for model-related errors."""
    pass

class TrainingError(ModelError):
    def __init__(self, epoch, loss, message="Training failed"):
        self.epoch = epoch
        self.loss = loss
        super().__init__(f"{message} at epoch {epoch}: loss={loss:.4f}")

class InferenceError(ModelError):
    def __init__(self, input_shape, message="Inference failed"):
        self.input_shape = input_shape
        super().__init__(f"{message} for input shape {input_shape}")

# Usage
try:
    raise TrainingError(epoch=42, loss=nan, message="NaN loss detected")
except TrainingError as e:
    print(f"Caught: {e}")
    print(f"Epoch: {e.epoch}, Loss: {e.loss}")
    print(f"Is ModelError? {isinstance(e, ModelError)}")

> **Best Practice:** Create a base exception for your module/package, then specific subclasses. This allows callers to catch all your errors with `except ModelError`.

---

## 5. Context Managers (`with` statement)

In [ ]:
# Built-in context managers
with open("test.txt", "w") as f:
    f.write("hello")
# File automatically closed here, even if exception occurs

# Multiple context managers
with open("a.txt", "w") as f1, open("b.txt", "w") as f2:
    f1.write("a")
    f2.write("b")

### Creating Custom Context Managers

#### Approach 1: Class with `__enter__`/`__exit__`

In [ ]:
import time

class Timer:
    def __enter__(self):
        self.start = time.perf_counter()
        return self  # Returned as 'as timer'
    
    def __exit__(self, exc_type, exc_val, exc_tb):
        self.end = time.perf_counter()
        self.elapsed = self.end - self.start
        print(f"Elapsed: {self.elapsed:.4f}s")
        # Return True to suppress exception, False/None to propagate
        return False

with Timer() as t:
    sum(range(10_000_000))
    # print(t.elapsed)  # Available after __exit__

#### Approach 2: `@contextmanager` Decorator (Simpler)

In [ ]:
from contextlib import contextmanager

@contextmanager
def timer():
    start = time.perf_counter()
    try:
        yield  # Code in 'with' block runs here
    finally:
        elapsed = time.perf_counter() - start
        print(f"Elapsed: {elapsed:.4f}s")

with timer():
    sum(range(10_000_000))

#### Approach 3: `contextlib.ExitStack` (Dynamic Number of Resources)

In [ ]:
from contextlib import ExitStack

filenames = ["a.txt", "b.txt", "c.txt"]

with ExitStack() as stack:
    files = [stack.enter_context(open(f, "w")) for f in filenames]
    for i, f in enumerate(files):
        f.write(f"file {i}")
# All files closed automatically

---

## 6. `contextlib` Utilities

In [ ]:
from contextlib import suppress, redirect_stdout, redirect_stderr, nullcontext
import io

# suppress: Ignore specific exceptions
with suppress(FileNotFoundError):
    open("nonexistent.txt").read()
print("Continued despite missing file")

# redirect_stdout/stderr: Capture print output
buf = io.StringIO()
with redirect_stdout(buf):
    print("captured")
    print("also captured")
print(f"Got: {buf.getvalue()}")

# nullcontext: No-op context manager (useful for conditional 'with')
debug = True
with (timer() if debug else nullcontext()):
    pass

---

## 7. `try/except` in Loops & Comprehensions

In [ ]:
# Pattern: Try in loop, continue on error
data = ["1", "2", "not_a_number", "4"]
results = []
for item in data:
    try:
        results.append(int(item))
    except ValueError:
        print(f"Skipping: {item}")
print(results)

# List comprehension with helper function
def safe_int(x):
    try:
        return int(x)
    except ValueError:
        return None

[safe_int(x) for x in data]

---

## 8. `assert` vs Exceptions

In [ ]:
# assert: For internal invariants, debugging, tests
# Can be disabled with: python -O (optimize mode)
def process_batch(batch):
    assert len(batch) > 0, "Batch cannot be empty"
    assert all(isinstance(x, (int, float)) for x in batch), "All numeric"
    return sum(batch) / len(batch)

# Exceptions: For expected runtime errors, user input, external systems
def load_model(path):
    if not path.exists():
        raise FileNotFoundError(f"Model not found: {path}")
    # ... load logic

> **Rule of Thumb:** Use `assert` for "this should never happen if code is correct" (programmer errors). Use exceptions for "this might happen in production" (runtime conditions).

---

## 9. Exception Groups (Python 3.11+)

In [ ]:
# Multiple exceptions at once (async, concurrent code)
try:
    raise ExceptionGroup("Multiple errors", [
        ValueError("bad value"),
        TypeError("bad type"),
        KeyError("missing key")
    ])
except* ValueError as eg:
    print(f"Caught ValueErrors: {eg.exceptions}")
except* TypeError as eg:
    print(f"Caught TypeErrors: {eg.exceptions}")

---

## 10. Best Practices Summary

In [ ]:
# DO:
try:
    result = api_call()
except (ConnectionError, TimeoutError) as e:
    logger.warning(f"API failed: {e}")
    raise ServiceUnavailable() from e

# DON'T:
try:
    result = api_call()
except:  # Bare except - catches KeyboardInterrupt!
    pass  # Silent failure - debugging nightmare

# DO: Use specific exceptions
try:
    value = config["key"]
except KeyError:
    value = default

# DO: Use else for success path
try:
    data = fetch()
except NetworkError:
    data = cache
else:
    cache.update(data)  # Only if fetch succeeded

# DO: Use finally for cleanup
lock.acquire()
try:
    critical_section()
finally:
    lock.release()  # Always released

---

## 11. Interview Questions

1. **What's the difference between `except Exception` and `except BaseException`?**
   - `BaseException` catches `KeyboardInterrupt`, `SystemExit`, `GeneratorExit` — rarely what you want

2. **When does `else` block execute in try/except?**
   - Only if NO exception was raised in `try`

3. **What does `finally` do? When does it NOT run?**
   - Always runs (cleanup). Doesn't run only if process is killed (`os._exit`, `SIGKILL`)

4. **How do you preserve the original traceback when re-raising?**
   - Use bare `raise` (not `raise e` which resets traceback)

5. **What's exception chaining? (`from e`)**
   - Links cause (`__cause__`) or context (`__context__`) for debugging

6. **Why use context managers over try/finally?**
   - Guaranteed cleanup, readable, composable, reusable

7. **What does `__exit__` return value mean?**
   - `True` = suppress exception, `False/None` = propagate

8. **Difference between `assert` and `raise`?**
   - `assert`: debugging, can be disabled with `-O`, for invariants
   - `raise`: production error handling, always runs

9. **How to handle multiple exceptions in concurrent code (3.11+)?**
   - `ExceptionGroup` and `except*` syntax

10. **What's the MRO for exception lookup?**
    - Same as attribute lookup: instance → class → base classes → `object`